<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/GaussianGPT_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧊 GaussianGPT — Autoregressive 3D Gaussian Scene Generation (ECCV 2026, MIT)

A Colab port of [GaussianGPT](https://github.com/nicolasvonluetzow/GaussianGPT) (Nicolas von Lützow, Barbara Rössle, Katharina Schmid, Matthias Nießner — ECCV 2026). GaussianGPT generates **3D Gaussian scenes completely autoregressively** via next-token prediction — the same paradigm as GPT-style text models, but applied to 3D Gaussian splats.

## What it does

Unlike diffusion-based 3D generation methods that refine scenes holistically, GaussianGPT constructs scenes **step-by-step**:

1. **VQ-VAE** compresses per-voxel Gaussians into discrete tokens (a sparse 3D CNN with vector quantization, supervised by a gsplat re-rendering loss)
2. **GPT** models the token sequences autoregressively with 3D rotary positional embeddings
3. **Decode** sampled tokens back through the frozen VQ-VAE to get renderable Gaussian splats
4. **Render** the splats to orbit GIFs or top-down PNGs

```
Gaussians --VQ-VAE--> tokens --GPT--> sampled tokens --decode--> Gaussians --render-->
```

This formulation naturally supports:
- **Unconditional generation** — sample new scenes from scratch
- **Completion** — keep part of a scene, let the model fill in the rest
- **Outpainting** — extend scenes spatially (multi-chunk tiling)
- **Controllable sampling** — temperature, top-k, top-p

## Two model variants

| Variant | Grid | GPT size | Checkpoints | Use case |
|---|---|---|---|---|
| **Scene-level (VFront)** | 20³ voxels | GPT-2-medium (24L, 1024d, ≈350M) | vqvae_vfront + gpt_vfront (≈5.5 GB) | Indoor room scenes (3D-FRONT) |
| **Object-level (PhotoShape)** | 32³ voxels | GPT-2-small (12L, 768d, ≈100M) | vqvae_photoshape + gpt_photoshape (≈3.1 GB) | Individual objects |

## Quick start

1. **Runtime → Change runtime type → GPU** (L4 or A100 recommended; T4 may be tight).
2. Run **STEP 1** — installs torch 2.9.0+cu128, MinkowskiEngine (pre-built wheel), gsplat + pytorch3d from source, remaining deps. First run: ≈25-35 min (gsplat + pytorch3d compilation are the slow parts).
3. Run **STEP 2** — downloads checkpoints (≈5.5 GB scene-level or ≈3.1 GB object-level) to Drive cache.
4. Run **STEP 4** — opens the Gradio UI. Pick a model variant, set temperature/top-k/top-p, click **Generate**.
5. **STEP 6** gives a single-sample quick test; **STEP 7** generates multiple scenes in batch.

## Outputs

Everything is written to `AEI_3D_Cache/GaussianGPT/samples/`:

```
sample_0000.pt          # Gaussian payload (means, sh0, opacities, scales, quats)
sample_0000.gif         # Orbit render (120 frames @ 24 fps)
sample_0000.ply         # INRIA-style 3DGS .ply (SuperSplat / PlayCanvas compatible)
```

## Technical notes

- **MinkowskiEngine**: Uses the [alpsaur fork](https://github.com/alpsaur/MinkowskiEngine) v0.5.8 pre-built wheel (9.6 MB, cp312, torch 2.9.x+cu128). No compilation needed — the biggest dependency risk is eliminated.
- **gsplat + pytorch3d**: Built from source against pinned git commits. ≈10-15 min each on L4.
- **Flash Attention**: NOT required. The code has a built-in SDPA fallback for non-Hopper GPUs (L4 = sm_89, not Hopper). On H100 (sm_90) it would use FA3 automatically.
- **torch 2.9.0+cu128**: Matches the MinkowskiEngine wheel's build version. Force-reinstalled over whatever Colab ships.

## License

MIT License (Copyright © 2026 Nicolas von Lützow). Fully compatible with our suite.

## Companion notebooks

- **MapAnything_Colab** — Apache 2.0, single-image → 3D in ≈10 s
- **NoPoSplat_Colab** — MIT, 2-3 photos → 3DGS in ≈10 s
- **HY-World-2.0_Colab** — Tencent, multi-view → 3DGS + GLB + depth + normals
- **TripoSplat_Colab** — MIT, text/image → 3DGS
- **SplatTransform_Colab** — 3DGS format converter + voxel collision mesh
- **TextureMapPrep_Colab** — seamless PBR maps for game assets


In [ ]:
#@title STEP 1 — Install torch 2.9.0+cu128, MinkowskiEngine wheel, gsplat + pytorch3d from source
"""
• Pins torch to 2.9.0+cu128 (matches the alpsaur MinkowskiEngine pre-built wheel)
• Installs MinkowskiEngine from the alpsaur fork v0.5.8 pre-built cp312 wheel (9.6 MB, no compilation)
• Builds gsplat from a pinned git commit (~10-15 min on L4)
• Builds pytorch3d from a pinned git commit (~10-15 min on L4)
• Installs remaining pip deps (lightning, hydra, vector-quantize-pytorch, etc.)
• Clones the GaussianGPT repo
• Flash Attention is NOT needed — the code has a built-in SDPA fallback for non-Hopper GPUs
"""
import os, sys, time, subprocess, shutil, pathlib, importlib

print('='*72)
print('GaussianGPT — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — inference will be very slow')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/GaussianGPT')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Drive cache  : {drive_root}')
else:
    drive_root = pathlib.Path('/content/_ggpt_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}')

OUT_DIR = drive_root / 'samples'
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT = pathlib.Path('/content/ggpt_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_ROOT / 'GaussianGPT'
CKPT_DIR = drive_root / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

t_total = time.time()

# 1. Pin torch to 2.9.0+cu128 (matches ME wheel build version) ──────────
TARGET_TORCH = '2.9.0'
if not torch.__version__.startswith(TARGET_TORCH):
    print(f'\n[1/6] Pinning torch to {TARGET_TORCH}.0+cu128 ...')
    t0 = time.time()
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--disable-pip-version-check', '--no-input',
        f'torch=={TARGET_TORCH}.0+cu128',
        'torchvision==0.24.0+cu128',
        'torchaudio==2.9.0+cu128',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
        '--force-reinstall',
    ], check=False)
    print(f'  torch pinned in {time.time()-t0:.1f}s')
    importlib.reload(importlib.import_module('torch'))
    import torch
    print(f'  torch now    : {torch.__version__}  (CUDA {torch.version.cuda})')
else:
    print(f'\n[1/6] torch {torch.__version__} already matches {TARGET_TORCH} — skipping')

# 2. MinkowskiEngine pre-built wheel (alpsaur v0.5.8) ───────────────────
print('\n[2/6] Installing MinkowskiEngine (alpsaur v0.5.8 pre-built wheel) ...')
t0 = time.time()
ME_WHEEL = (
    'https://github.com/alpsaur/MinkowskiEngine/releases/download/v0.5.8/'
    f'minkowskiengine-0.5.8-cp{sys.version_info.major}{sys.version_info.minor}'
    f'-cp{sys.version_info.major}{sys.version_info.minor}-linux_x86_64.whl'
)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', ME_WHEEL], check=False)
try:
    import MinkowskiEngine as ME
    print(f'  MinkowskiEngine {ME.__version__} installed in {time.time()-t0:.1f}s')
except ImportError as e:
    print(f'  [WARN] MinkowskiEngine import failed: {e}')
    print('  Falling back to building from source (10-20 min) ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja'], check=False)
    ME_SRC = WORK_ROOT / 'MinkowskiEngine'
    if ME_SRC.exists():
        shutil.rmtree(ME_SRC, ignore_errors=True)
    subprocess.run(['git', 'clone', '--quiet', '--depth=1',
                     '--branch', 'cuda12-compat',
                     'https://github.com/alpsaur/MinkowskiEngine.git', str(ME_SRC)],
                   capture_output=True)
    subprocess.run([sys.executable, 'setup.py', 'install'],
                   cwd=str(ME_SRC), capture_output=True)
    import MinkowskiEngine as ME
    print(f'  MinkowskiEngine {ME.__version__} built from source in {time.time()-t0:.1f}s')

# 3. gsplat from source (pinned git commit) ─────────────────────────────
print('\n[3/6] Building gsplat from source (pinned commit) ...')
t0 = time.time()
GSPLAT_COMMIT = 'd23d7ca5dd26c3756967304b621ae88521672ed5'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
    f'git+https://github.com/nerfstudio-project/gsplat.git@{GSPLAT_COMMIT}',
], check=False)
try:
    import gsplat
    print(f'  gsplat {gsplat.__version__} built in {time.time()-t0:.1f}s')
except ImportError as e:
    print(f'  [WARN] gsplat import failed: {e}')

# 4. pytorch3d from source (pinned git commit) ──────────────────────────
print('\n[4/6] Building pytorch3d from source (pinned commit) ...')
t0 = time.time()
P3D_COMMIT = '75ebeeaea0908c5527e7b1e305fbc7681382db47'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
    f'git+https://github.com/facebookresearch/pytorch3d.git@{P3D_COMMIT}',
], check=False)
try:
    import pytorch3d
    print(f'  pytorch3d {pytorch3d.__version__} built in {time.time()-t0:.1f}s')
except ImportError as e:
    print(f'  [WARN] pytorch3d import failed: {e}')

# 5. Remaining pip deps ──────────────────────────────────────────────────
print('\n[5/6] Installing remaining pip deps ...')
t0 = time.time()
EXTRA_PKGS = [
    'gradio>=5.49.1,<7',
    'ninja', 'plyfile', 'einops', 'lightning==2.5.2', 'tensorboard',
    'vector-quantize-pytorch', 'tqdm', 'torchmetrics[image]',
    'hydra-core', 'imageio', 'opencv-python-headless',
    'scikit-image', 'iopath', 'matplotlib', 'lpips',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_PKGS, check=False)
print(f'  Pip deps installed in {time.time()-t0:.1f}s')

# 6. Clone GaussianGPT repo ─────────────────────────────────────────────
print('\n[6/6] Cloning GaussianGPT repo ...')
t0 = time.time()
if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print('  Already cloned — pulling latest')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--quiet', '--ff-only'],
                   capture_output=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--quiet', '--depth=1',
                     'https://github.com/nicolasvonluetzow/GaussianGPT.git', str(REPO_DIR)],
                   check=True)
sys.path.insert(0, str(REPO_DIR))
print(f'  GaussianGPT cloned in {time.time()-t0:.1f}s')

elapsed = time.time() - t_total
print()
print('='*72)
print(f'STEP 1 complete in {elapsed/60:.1f} min')
print('='*72)
print(f'  Drive cache   : {drive_root}')
print(f'  Repo          : {REPO_DIR}')
print(f'  Checkpoint dir: {CKPT_DIR}')
print()
print('Next: run STEP 2 (download checkpoints).')


In [ ]:
#@title STEP 2 — Download checkpoints to Drive cache
"""
Downloads the pre-trained VQ-VAE and GPT checkpoints from
kaldir.vc.cit.tum.de/gaussiangpt/ into the Drive cache.

Two variants are available:
  - Scene-level (VFront): vqvae_vfront.ckpt (2.1 GB) + gpt_vfront.ckpt (3.4 GB) = ~5.5 GB
  - Object-level (PhotoShape): vqvae_photoshape.ckpt (1.3 GB) + gpt_photoshape.ckpt (1.8 GB) = ~3.1 GB

The server requires a User-Agent header (default curl/Python UA gets 403).
We use wget with a custom UA which works reliably.
"""
import os, sys, time, subprocess, pathlib

print('='*72)
print('GaussianGPT — Checkpoint download')
print('='*72)

CKPT_BASE = 'https://kaldir.vc.cit.tum.de/gaussiangpt'

# Checkpoint registry: (filename, size_gb, description)
CHECKPOINTS = {
    'scene_vfront': {
        'vqvae_vfront.ckpt': 2.1,
        'gpt_vfront.ckpt': 3.4,
    },
    'object_photoshape': {
        'vqvae_photoshape.ckpt': 1.3,
        'gpt_photoshape.ckpt': 1.8,
    },
}

MODEL_VARIANT = 'scene_vfront'  #@param ['scene_vfront', 'object_photoshape', 'both']

print(f'  Model variant : {MODEL_VARIANT}')
print(f'  Cache dir     : {CKPT_DIR}')
print()

if MODEL_VARIANT == 'both':
    to_download = list(CHECKPOINTS['scene_vfront'].keys()) + list(CHECKPOINTS['object_photoshape'].keys())
    total_gb = sum(CHECKPOINTS['scene_vfront'].values()) + sum(CHECKPOINTS['object_photoshape'].values())
else:
    to_download = list(CHECKPOINTS[MODEL_VARIANT].keys())
    total_gb = sum(CHECKPOINTS[MODEL_VARIANT].values())

print(f'  Downloading {len(to_download)} file(s), ~{total_gb:.1f} GB total')
print()

t_total = time.time()
for fname in to_download:
    dst = CKPT_DIR / fname
    if dst.exists() and dst.stat().st_size > 100_000_000:
        print(f'  [skip] {fname} already cached ({dst.stat().st_size/1024**3:.1f} GB)')
        continue
    print(f'  Downloading {fname} ...')
    t0 = time.time()
    r = subprocess.run([
        'wget', '-q', '--no-check-certificate',
        '-U', 'Wget/1.21',
        '-O', str(dst),
        f'{CKPT_BASE}/{fname}',
    ], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'    [ERROR] wget failed: {r.stderr[-500:]}')
        # Try curl as fallback
        r2 = subprocess.run([
            'curl', '-sL', '--retry', '3',
            '-H', 'User-Agent: Wget/1.21',
            '-o', str(dst),
            f'{CKPT_BASE}/{fname}',
        ], capture_output=True, text=True)
        if r2.returncode != 0:
            print(f'    [ERROR] curl also failed: {r2.stderr[-500:]}')
            raise RuntimeError(f'Failed to download {fname}')
    if dst.exists():
        sz = dst.stat().st_size / 1024**3
        print(f'    OK ({sz:.1f} GB in {time.time()-t0:.0f}s)')
    else:
        print(f'    [ERROR] file not found after download')
        raise RuntimeError(f'Failed to download {fname}')

elapsed = time.time() - t_total
print()
print('='*72)
print(f'STEP 2 complete in {elapsed:.0f}s')
print('='*72)
print(f'  Checkpoints cached at: {CKPT_DIR}')
for f in CKPT_DIR.glob('*.ckpt'):
    print(f'    {f.name}  ({f.stat().st_size/1024**3:.1f} GB)')
print()
print('Next: run STEP 3 (imports + lazy model loader).')


In [ ]:
#@title STEP 3 — Imports, lazy model loader, render + PLY helpers
"""
• Verifies all deps are importable
• Defines `load_gaussiangpt()` — the lazy loader that picks the right
  checkpoint pair based on MODEL_VARIANT, loads the VQ-VAE + GPT,
  and returns (gpt, vqvae, device)
• Defines `render_scene_to_gif()` and `save_scene_as_ply()` helpers
• Defines `sample_one()` — a convenience wrapper around gpt.sample()
  that returns a GaussianScene + renders a GIF + saves a .pt + .ply
"""
import os, sys, time, json, gc, pathlib, traceback
import torch
import numpy as np
from PIL import Image

print('='*72)
print('GaussianGPT — Imports + lazy loader')
print('='*72)

# --- Verify deps ────────────────────────────────────────────────────────
deps_ok = True
for mod_name in ['torch', 'gsplat', 'MinkowskiEngine', 'lightning', 'hydra',
                  'vector_quantize_pytorch', 'einops', 'plyfile', 'imageio']:
    try:
        __import__(mod_name)
    except ImportError as e:
        print(f'  [FAIL] {mod_name}: {e}')
        deps_ok = False
if not deps_ok:
    raise RuntimeError('Missing dependencies — re-run STEP 1')

import lightning
import MinkowskiEngine as ME
import gsplat
print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda})')
print(f'  lightning    : {lightning.__version__}')
print(f'  gsplat       : {gsplat.__version__}')
print(f'  MinkowskiEngine: {ME.__version__}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU          : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
else:
    print('  WARNING: no GPU detected')
print()

# --- Add repo to path ──────────────────────────────────────────────────
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- Lazy model loader (singleton) ────────────────────────────────────
_GPT = None
_VQVAE = None
_DEVICE = None
_VARIANT = None

def load_gaussiangpt(variant='scene_vfront', verbose=True):
    """Load the GaussianGPT model from cached checkpoints.

    Args:
        variant: 'scene_vfront' or 'object_photoshape'
    Returns: (gpt, vqvae, device)
    """
    global _GPT, _VQVAE, _DEVICE, _VARIANT
    if _GPT is not None and _VARIANT == variant:
        return _GPT, _VQVAE, _DEVICE

    from model.gaussian_gpt import GaussianGPT
    from model.gaussian_vqvae import GaussianVQVAE

    if variant == 'scene_vfront':
        vqvae_path = str(CKPT_DIR / 'vqvae_vfront.ckpt')
        gpt_path = str(CKPT_DIR / 'gpt_vfront.ckpt')
    elif variant == 'object_photoshape':
        vqvae_path = str(CKPT_DIR / 'vqvae_photoshape.ckpt')
        gpt_path = str(CKPT_DIR / 'gpt_photoshape.ckpt')
    else:
        raise ValueError(f'Unknown variant: {variant}')

    for p in [vqvae_path, gpt_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f'Checkpoint not found: {p}. Run STEP 2 first.')

    _DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if verbose:
        print(f'  Loading VQ-VAE from {vqvae_path} ...')
    t0 = time.time()
    _VQVAE = GaussianVQVAE.load_from_checkpoint(vqvae_path).eval()
    if verbose:
        print(f'  VQ-VAE loaded in {time.time()-t0:.1f}s')

    if verbose:
        print(f'  Loading GPT from {gpt_path} ...')
    t0 = time.time()
    _GPT = GaussianGPT.load_from_checkpoint(gpt_path, vqvae=_VQVAE).eval().to(_DEVICE)
    if verbose:
        n_params = sum(p.numel() for p in _GPT.parameters())
        print(f'  GPT loaded in {time.time()-t0:.1f}s ({n_params/1e6:.1f}M params)')
    _VARIANT = variant
    return _GPT, _VQVAE, _DEVICE

def free_gaussiangpt():
    """Unload the model and free GPU memory."""
    global _GPT, _VQVAE, _DEVICE, _VARIANT
    if _GPT is not None:
        del _GPT
    if _VQVAE is not None:
        del _VQVAE
    _GPT = None
    _VQVAE = None
    _DEVICE = None
    _VARIANT = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Render helper ──────────────────────────────────────────────────────
def render_scene_to_gif(scene, output_path, num_frames=120, fps=24,
                         background_color='white', up='z'):
    """Render a GaussianScene to an orbit GIF using gsplat."""
    from utils.render import (render_and_save_trajectory, center_scene_aabb,
                              rotate_scene_y_up_to_z_up)
    render_scene = center_scene_aabb(scene)
    if up == 'y':
        # PhotoShape is y-up; rotate to z-up using the upstream helper
        # which correctly rotates means + quats + degree-1 SH.
        render_scene = rotate_scene_y_up_to_z_up(render_scene)
    render_and_save_trajectory(
        render_scene,
        str(output_path),
        num_frames=num_frames,
        fps=fps,
        background_color=background_color,
    )

# --- PLY conversion helper ─────────────────────────────────────────────
def save_scene_as_ply(scene, output_path):
    """Convert a GaussianScene to an INRIA-style 3DGS .ply file."""
    from data.photoshape import save_inria_ply
    d = scene.to_dict()
    save_inria_ply(
        str(output_path),
        coords=d['means'],
        sh0=d['sh0'],
        opacities=d['opacities'],
        scales=d['scales'],
        quats=d['quats'],
        sh=d.get('sh'),
    )

# --- Sample helper ─────────────────────────────────────────────────────
def sample_one(gpt, temperature=0.9, top_k=None, top_p=0.9, seed=42,
               render_gif=True, save_pt=True, save_ply=True,
               output_dir=None, sample_idx=0, gif_frames=120, gif_fps=24,
               background_color='white', up='z'):
    """Sample one scene from the GPT, render + save.

    Returns: dict with 'scene', 'gif_path', 'pt_path', 'ply_path'
    """
    t0 = time.time()
    scenes, lengths = gpt.sample(
        num_samples=1,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        return_lengths=True,
        seed=seed,
    )
    sample_time = time.time() - t0
    scene = scenes[0]
    result = {
        'scene': scene,
        'sample_time': sample_time,
        'token_length': int(lengths[0].item()) if lengths is not None else 0,
        'gif_path': None,
        'pt_path': None,
        'ply_path': None,
    }

    if output_dir is None:
        output_dir = OUT_DIR
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if render_gif:
        gif_path = output_dir / f'sample_{sample_idx:04d}.gif'
        try:
            render_scene_to_gif(scene, gif_path, num_frames=gif_frames,
                                fps=gif_fps, background_color=background_color, up=up)
            result['gif_path'] = str(gif_path)
        except Exception as e:
            print(f'  [WARN] render failed: {e}')

    if save_pt:
        pt_path = output_dir / f'sample_{sample_idx:04d}.pt'
        payload = {
            k: v.detach().cpu() if torch.is_tensor(v) else v
            for k, v in scene.to_dict().items()
        }
        torch.save(payload, pt_path)
        result['pt_path'] = str(pt_path)

    if save_ply:
        ply_path = output_dir / f'sample_{sample_idx:04d}.ply'
        try:
            save_scene_as_ply(scene, ply_path)
            result['ply_path'] = str(ply_path)
        except Exception as e:
            print(f'  [WARN] PLY conversion failed: {e}')

    return result

# --- CUDA free helper ──────────────────────────────────────────────────
def free_cuda(verbose=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            free, total = torch.cuda.mem_get_info()
            print(f'  [cuda] free={free/1024**3:.1f} GB / total={total/1024**3:.1f} GB')

print('STEP 3 complete — model loader + helpers ready.')
print('Next: run STEP 4 to open the Gradio UI.')


In [ ]:
#@title STEP 4 — Gradio UI (generate, completion, batch)
"""
• Two-column layout: left = controls, right = output gallery + GIF viewer
• Model variant selector (scene_vfront / object_photoshape)
• Sampling controls: temperature, top_k, top_p, num_samples, seed
• Render controls: gif_frames, gif_fps, background_color
• Download buttons for .ply and .pt
• Concurrency limit = 2 (matches the rest of the AEI suite)
• clear_output(wait=True) before launch
• demo.load welcome message
"""
import os, sys, time, json, gc, pathlib, traceback
import torch
import numpy as np
import gradio as gr

# --- Gradio UI ─────────────────────────────────────────────────────────
CSS = """
#col-container   { margin: 0 auto; max-width: 1400px; }
#main-title h1   { font-size: 2.4em !important; }
"""

with gr.Blocks(css=CSS, delete_cache=(600, 600)) as demo:
    gr.Markdown(
        '# **GaussianGPT — Autoregressive 3D Gaussian Scene Generation**',
        elem_id='main-title',
    )
    gr.Markdown(
        'Generate 3D Gaussian scenes autoregressively via next-token prediction. '
        'Powered by [GaussianGPT](https://github.com/nicolasvonluetzow/GaussianGPT) (ECCV 2026, MIT).'
    )

    with gr.Row(elem_id='col-container'):
        with gr.Column(scale=1, min_width=380):
            model_variant = gr.Radio(
                choices=['scene_vfront', 'object_photoshape'],
                value='scene_vfront',
                label='Model variant',
                info='Scene-level generates indoor rooms (3D-FRONT). Object-level generates individual objects (PhotoShape).',
            )
            with gr.Accordion('Sampling parameters', open=True):
                temperature = gr.Slider(
                    0.1, 2.0, value=0.9, step=0.05,
                    label='Temperature',
                    info='Higher = more diverse, lower = more conservative. 0.9 is the default.',
                )
                top_k = gr.Slider(
                    0, 200, value=0, step=1,
                    label='Top-k (0 = off)',
                    info='Restrict sampling to the top-k tokens. 0 = no restriction.',
                )
                top_p = gr.Slider(
                    0.1, 1.0, value=0.9, step=0.05,
                    label='Top-p (nucleus)',
                    info='Restrict sampling to the top-p probability mass. 0.9 is the default.',
                )
                num_samples = gr.Slider(
                    1, 16, value=4, step=1,
                    label='Number of samples',
                    info='How many scenes to generate in one batch.',
                )
                seed = gr.Number(
                    value=42,
                    label='Random seed',
                    info='Set for reproducible results. Different seeds = different scenes.',
                    precision=0,
                )
            with gr.Accordion('Render options', open=False):
                gif_frames = gr.Slider(
                    30, 360, value=120, step=10,
                    label='GIF frames',
                    info='Number of orbit frames. More = smoother but larger file.',
                )
                gif_fps = gr.Slider(
                    6, 60, value=24, step=2,
                    label='GIF FPS',
                    info='Playback speed. 24 fps is the default.',
                )
                background_color = gr.Radio(
                    choices=['white', 'black'],
                    value='white',
                    label='Background color',
                    info='Background for rendered GIFs.',
                )
                save_ply = gr.Checkbox(
                    value=True,
                    label='Save .ply (SuperSplat compatible)',
                    info='Convert each sample to an INRIA-style 3DGS .ply file.',
                )
                save_pt = gr.Checkbox(
                    value=True,
                    label='Save .pt (raw Gaussian payload)',
                    info='Save the raw Gaussian scene data for later rendering or conversion.',
                )
            btn_generate = gr.Button('Generate scenes', variant='primary')
            status_box = gr.Textbox(
                label='Status', interactive=False, lines=3,
                placeholder='Awaiting generation...',
            )
            with gr.Accordion('Downloads', open=False):
                dl_ply = gr.File(label='Download .ply files', file_count='multiple')
                dl_gif = gr.File(label='Download .gif files', file_count='multiple')

        with gr.Column(scale=2):
            gr.Markdown('### Generated scenes')
            output_gallery = gr.Gallery(
                label='Orbit GIFs',
                columns=2, rows=2, height=500,
                object_fit='contain', preview=True,
            )
            gr.Markdown(
                'Each GIF is an orbit render of a generated 3D Gaussian scene. '
                'The .ply files can be opened in [SuperSplat](https://supersplat.xyz) or PlayCanvas.'
            )

    # --- Event wiring ---
    def generate(variant, temp, k, p, n, s, gif_f, gif_fps_val, bg, do_ply, do_pt):
        try:
            gpt, vqvae, device = load_gaussiangpt(variant=variant)
            up = 'y' if variant == 'object_photoshape' else 'z'
            k_val = int(k) if k > 0 else None
            output_subdir = OUT_DIR / f'gen_{int(time.time())}'
            output_subdir.mkdir(parents=True, exist_ok=True)

            t0 = time.time()
            results = []
            gif_paths = []
            ply_paths = []
            for i in range(int(n)):
                r = sample_one(
                    gpt, temperature=float(temp), top_k=k_val, top_p=float(p),
                    seed=int(s) + i, render_gif=True, save_pt=do_pt, save_ply=do_ply,
                    output_dir=output_subdir, sample_idx=i,
                    gif_frames=int(gif_f), gif_fps=int(gif_fps_val),
                    background_color=bg, up=up,
                )
                results.append(r)
                if r['gif_path']:
                    gif_paths.append(r['gif_path'])
                if r['ply_path']:
                    ply_paths.append(r['ply_path'])
                free_cuda()

            elapsed = time.time() - t0
            status = (
                f'Generated {len(results)} scene(s) in {elapsed:.1f}s. '
                f'Output: {output_subdir.name}'
            )
            gallery_items = [(p, pathlib.Path(p).stem) for p in gif_paths] if gif_paths else []
            return (
                gr.update(value=gallery_items),
                gr.update(visible=True, value=status),
                gr.update(value=ply_paths) if ply_paths else gr.update(),
                gr.update(value=gif_paths) if gif_paths else gr.update(),
            )
        except Exception as e:
            traceback.print_exc(limit=4)
            raise gr.Error(f'Generation failed: {e}')

    btn_generate.click(
        generate,
        inputs=[model_variant, temperature, top_k, top_p, num_samples, seed,
                gif_frames, gif_fps, background_color, save_ply, save_pt],
        outputs=[output_gallery, status_box, dl_ply, dl_gif],
    )

    def _welcome():
        return (
            'Select a model variant, adjust sampling parameters, then click '
            '"Generate scenes". First run loads the checkpoints (~30s after STEP 2).'
        )
    demo.load(_welcome, inputs=None, outputs=[status_box])

# --- Queue + launch ────────────────────────────────────────────────────
demo.queue(concurrency_limit=2, max_size=8)
try:
    from IPython.display import clear_output
    clear_output()
    clear_output(wait=True)
except Exception:
    pass
demo.launch(share=False, server_name='0.0.0.0', server_port=7860, show_error=True, height=1100)


In [ ]:
#@title STEP 5 — Keep alive + session summary
"""
Standard AEI-suite keep-alive cell.  Hits a local URL once a minute via
IPython.display.Javascript to prevent the 90-minute idle disconnect.
"""
import os, sys, time, pathlib, datetime
import IPython
from IPython.display import display, Javascript

print('='*72)
print('Keep-alive timer started.')
print('='*72)

try:
    summary = {
        'cache_root'    : str(drive_root),
        'repo_dir'      : str(REPO_DIR),
        'ckpt_dir'      : str(CKPT_DIR),
        'out_dir'       : str(OUT_DIR),
        'torch'         : torch.__version__,
        'cuda'          : torch.version.cuda,
        'gsplat'        : gsplat.__version__,
        'MinkowskiEngine': ME.__version__,
        'gpu'           : None,
    }
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        summary['gpu'] = f'{p.name}  ({p.total_memory / (1024**3):.1f} GB)'
    print('\n  Session summary')
    print('  ' + '-'*68)
    for k, v in summary.items():
        print(f'  {k:18s}: {v}')
except Exception as e:
    print(f'  WARN: summary print failed: {e}')

display(Javascript('''
function ClickConnect() {
  console.log("Keeping Colab alive — ", new Date().toLocaleTimeString());
  document.querySelector("colab-connect-button")?.click();
}
setInterval(ClickConnect, 60000);
'''))
print('\n  Keep-alive timer registered (60 s interval).')


In [ ]:
#@title STEP 6 — Quick test (single sample generation)
"""
Stand-alone test path.  Generates one scene with default parameters,
renders a GIF, and displays a FileLink for download.
"""
import os, sys, time, pathlib, traceback
import torch
from IPython.display import display, FileLink, Image as IPImage

print('='*72)
print('GaussianGPT — single-sample quick test')
print('='*72)

MODEL_VARIANT = 'scene_vfront'  #@param ['scene_vfront', 'object_photoshape']
TEMPERATURE = 0.9  #@param {type:"slider", min:0.1, max:2.0, step:0.05}
TOP_K = 0  #@param {type:"slider", min:0, max:200, step:1}
TOP_P = 0.9  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
SEED = 42  #@param {type:"integer"}

# Load model
gpt, vqvae, device = load_gaussiangpt(variant=MODEL_VARIANT)
up = 'y' if MODEL_VARIANT == 'object_photoshape' else 'z'
k_val = int(TOP_K) if TOP_K > 0 else None

output_subdir = OUT_DIR / f'quicktest_{int(time.time())}'
output_subdir.mkdir(parents=True, exist_ok=True)

print(f'  Variant    : {MODEL_VARIANT}')
print(f'  Temperature: {TEMPERATURE}')
print(f'  Top-k      : {k_val}')
print(f'  Top-p      : {TOP_P}')
print(f'  Seed       : {SEED}')
print(f'  Output     : {output_subdir}')
print()

t0 = time.time()
result = sample_one(
    gpt, temperature=TEMPERATURE, top_k=k_val, top_p=TOP_P, seed=SEED,
    render_gif=True, save_pt=True, save_ply=True,
    output_dir=output_subdir, sample_idx=0,
    gif_frames=120, gif_fps=24, background_color='white', up=up,
)
elapsed = time.time() - t0

print()
print('='*72)
print(f'Sample generated in {elapsed:.1f}s')
print(f'  Token length: {result["token_length"]}')
print(f'  GIF         : {result["gif_path"]}')
print(f'  PT          : {result["pt_path"]}')
print(f'  PLY         : {result["ply_path"]}')
print('='*72)

# Show file sizes
for label, path_key in [('GIF', 'gif_path'), ('PT', 'pt_path'), ('PLY', 'ply_path')]:
    p = result.get(path_key)
    if p and os.path.exists(p):
        sz = os.path.getsize(p)
        if sz > 1024 * 1024:
            sz_s = f'{sz/1024/1024:.1f} MB'
        elif sz > 1024:
            sz_s = f'{sz/1024:.1f} KB'
        else:
            sz_s = f'{sz} B'
        print(f'  {label:>4s}: {sz_s:>10s}  {pathlib.Path(p).name}')

# Display FileLink for the .ply
if result['ply_path']:
    print()
    display(FileLink(result['ply_path'], result_html_prefix='Download .ply: '))
if result['gif_path']:
    print()
    display(FileLink(result['gif_path'], result_html_prefix='Download .gif: '))

free_cuda()
print('\nSTEP 6 complete. Open the Gradio UI (STEP 4) for the full experience.')


In [ ]:
#@title STEP 7 — Batch generation
"""
Generate multiple scenes in batch.  Each scene is saved as .pt + .ply + .gif
to the output directory.  A progress log is written to batch_log.jsonl so
you can resume after a Colab disconnect.

The model is loaded once and reused for all samples.  GPU memory is cleared
between samples to prevent fragmentation.
"""
import os, sys, time, json, pathlib, traceback
import torch
from IPython.display import display, FileLink

print('='*72)
print('GaussianGPT — Batch generation')
print('='*72)

MODEL_VARIANT = 'scene_vfront'  #@param ['scene_vfront', 'object_photoshape']
NUM_SAMPLES = 8  #@param {type:"slider", min:1, max:64, step:1}
TEMPERATURE = 0.9  #@param {type:"slider", min:0.1, max:2.0, step:0.05}
TOP_K = 0  #@param {type:"slider", min:0, max:200, step:1}
TOP_P = 0.9  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
BASE_SEED = 42  #@param {type:"integer"}
GIF_FRAMES = 120  #@param {type:"slider", min:30, max:360, step:10}
GIF_FPS = 24  #@param {type:"slider", min:6, max:60, step:2}
BACKGROUND_COLOR = 'white'  #@param ['white', 'black']
SAVE_PLY = True  #@param {type:"boolean"}
SAVE_PT = True  #@param {type:"boolean"}

# Load model once
gpt, vqvae, device = load_gaussiangpt(variant=MODEL_VARIANT)
up = 'y' if MODEL_VARIANT == 'object_photoshape' else 'z'
k_val = int(TOP_K) if TOP_K > 0 else None

output_subdir = OUT_DIR / f'batch_{int(time.time())}'
output_subdir.mkdir(parents=True, exist_ok=True)
batch_log = output_subdir / 'batch_log.jsonl'

print(f'  Variant     : {MODEL_VARIANT}')
print(f'  Num samples : {NUM_SAMPLES}')
print(f'  Temperature : {TEMPERATURE}')
print(f'  Top-k       : {k_val}')
print(f'  Top-p       : {TOP_P}')
print(f'  Base seed   : {BASE_SEED}')
print(f'  Output      : {output_subdir}')
print()

results = []
batch_start = time.time()
log_f = open(batch_log, 'a', buffering=1)

for i in range(NUM_SAMPLES):
    seed = BASE_SEED + i
    print(f'  [{i+1:03d}/{NUM_SAMPLES}] seed={seed} ...', end='', flush=True)
    t0 = time.time()
    try:
        r = sample_one(
            gpt, temperature=TEMPERATURE, top_k=k_val, top_p=TOP_P, seed=seed,
            render_gif=True, save_pt=SAVE_PT, save_ply=SAVE_PLY,
            output_dir=output_subdir, sample_idx=i,
            gif_frames=GIF_FRAMES, gif_fps=GIF_FPS,
            background_color=BACKGROUND_COLOR, up=up,
        )
        elapsed = time.time() - t0
        print(f' OK ({elapsed:.1f}s, {r["token_length"]} tokens)')
        results.append(('ok', seed, r))
        log_f.write(json.dumps({
            'idx': i, 'seed': seed, 'status': 'ok',
            'elapsed_s': elapsed,
            'token_length': r['token_length'],
            'gif': r.get('gif_path'),
            'pt': r.get('pt_path'),
            'ply': r.get('ply_path'),
        }) + '\n')
    except Exception as e:
        elapsed = time.time() - t0
        print(f' FAIL ({elapsed:.1f}s): {e}')
        traceback.print_exc(limit=2)
        results.append(('error', seed, str(e)))
        log_f.write(json.dumps({
            'idx': i, 'seed': seed, 'status': 'error',
            'error': str(e), 'elapsed_s': elapsed,
        }) + '\n')
    free_cuda()

log_f.close()
total_elapsed = time.time() - batch_start
n_ok = sum(1 for r in results if r[0] == 'ok')
n_err = sum(1 for r in results if r[0] == 'error')

print()
print('='*72)
print(f'Batch complete: {n_ok} ok / {n_err} errors in {total_elapsed:.0f}s')
print(f'  Output: {output_subdir}')
print(f'  Log:   {batch_log}')
print('='*72)

# List all output files
for f in sorted(output_subdir.iterdir()):
    if f.is_file() and f.suffix in ('.gif', '.pt', '.ply'):
        sz = f.stat().st_size
        if sz > 1024 * 1024:
            sz_s = f'{sz/1024/1024:.1f} MB'
        elif sz > 1024:
            sz_s = f'{sz/1024:.1f} KB'
        else:
            sz_s = f'{sz} B'
        print(f'  {f.name:>30s}  {sz_s:>10s}')

if n_ok > 0:
    print(f'\n  Tip: zip with `!cd {output_subdir} && zip -r batch.zip .`')
    print(f'  Tip: open .ply files in [SuperSplat](https://supersplat.xyz)')
